# CTGAN — DGX training experiment

**Capstone:** Comparative evaluation of diffusion, GAN, and VAE models for synthetic clinical tabular data generation.

**Dataset:** Preprocessed UCI Diabetes 130-US Hospitals encounters. CTGAN is trained on the same fixed, patient-disjoint training split used for TVAE. The held-out test split is reserved for evaluation and is **not used for training**.

**Protocol:** Seed 42, explicit numerical/categorical metadata, 300 epochs, batch size 500, GPU training, and 79,567 synthetic rows (matching the training-set size). Results are saved under separate CTGAN filenames.

## 1. Runtime and reproducibility

Select the DGX `.venv` kernel. This cell selects the visible GPU, sets random seeds, finds the project folder, and checks that CTGAN output files do not already exist so earlier experiments cannot be overwritten. Run it in a fresh kernel before importing PyTorch elsewhere.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Set to your assigned DGX GPU before importing torch.

from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
import torch
import sdv

SEED = 42
EPOCHS = 300
BATCH_SIZE = 500

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/train_semantic.csv").is_file():
    ROOT = ROOT.parent
assert (ROOT / "data/processed/train_semantic.csv").is_file(), "Project training CSV not found"
assert torch.cuda.is_available(), "Select the DGX .venv kernel and check GPU assignment"

MODEL_PATH = ROOT / "models/ctgan_300ep.pkl"
CSV_PATH = ROOT / "data/synthetic/ctgan_300ep.csv"
METADATA_PATH = ROOT / "results/training/ctgan_300ep_metadata.json"
RUN_PATH = ROOT / "results/training/ctgan_300ep_run.json"
for path in [MODEL_PATH, CSV_PATH, METADATA_PATH, RUN_PATH]:
    assert not path.exists(), f"Output exists; will not overwrite: {path}"
    path.parent.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("SDV:", sdv.__version__)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

Project: /home/ayanasanthosh.ds25a/synthetic-clinical-data-generation
SDV: 1.38.5
PyTorch: 2.3.1+cu121
GPU: NVIDIA A100-SXM4-80GB


## 2. Training data and metadata

Load `train_semantic.csv` without modifying the held-out test data. The eight encounter counts/duration variables are numerical; all other columns, including the binary target `readmitted_30`, are categorical. Validate the metadata before model fitting.

In [2]:
from sdv.metadata import SingleTableMetadata

train = pd.read_csv(ROOT / "data/processed/train_semantic.csv", low_memory=False)
NUMERIC = [
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient", "number_diagnoses"
]
TARGET = "readmitted_30"

assert TARGET in train.columns
assert set(NUMERIC).issubset(train.columns)
assert not ({"patient_nbr", "encounter_id"} & set(train.columns))
assert not train.isna().any().any(), "Missing training values"

for col in train.columns:
    if col in NUMERIC:
        train[col] = pd.to_numeric(train[col], errors="raise")
    else:
        train[col] = train[col].astype(str)
assert set(train[TARGET].unique()) == {"0", "1"}

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=train)
for col in train.columns:
    metadata.update_column(
        column_name=col,
        sdtype="numerical" if col in NUMERIC else "categorical"
    )
metadata.validate()
metadata.validate_data(data=train)

print("Training shape:", train.shape)
print("Readmission labels:\n", train[TARGET].value_counts().sort_index())
print("Metadata validated:", len(metadata.to_dict()["columns"]), "columns")

Training shape: (79567, 45)
Readmission labels:
 readmitted_30
0    70497
1     9070
Name: count, dtype: int64
Metadata validated: 45 columns


## 3. CTGAN training

Train a fresh CTGAN model for **300 epochs** using the fixed training split and an available CUDA GPU. Record the wall-clock training time for later comparison with TVAE and TabDDPM.

In [ ]:
from sdv.single_table import CTGANSynthesizer

synthesizer = CTGANSynthesizer(
    metadata=metadata,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    enable_gpu=True,
    verbose=True,
)

start = time.perf_counter()
synthesizer.fit(train)
training_seconds = time.perf_counter() - start
print(f"CTGAN training completed in {training_seconds / 60:.2f} minutes")

/home/ayanasanthosh.ds25a/synthetic-clinical-data-generation/.venv/lib/python3.10/site-packages/sdv/single_table/base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/home/ayanasanthosh.ds25a/synthetic-clinical-data-generation/.venv/lib/python3.10/site-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (-04.91) | Discrim. (-00.01):  40%|████      | 120/300 [32:34<57:29, 19.16s/it] 

## 4. Generate and save synthetic data

Generate the same number of rows as the real training split. Save the model, synthetic CSV, metadata, and a run-summary JSON to distinct `ctgan_300ep` filenames. None of these outputs replaces the TVAE artifacts.

In [ ]:
synthetic_ctgan = synthesizer.sample(num_rows=len(train))
assert synthetic_ctgan.shape == train.shape, "Unexpected synthetic dataset shape"
assert set(synthetic_ctgan.columns) == set(train.columns), "Synthetic columns mismatch"

synthesizer.save(filepath=str(MODEL_PATH))
metadata.save_to_json(filepath=str(METADATA_PATH))
synthetic_ctgan.to_csv(CSV_PATH, index=False)

run_info = {
    "model": "CTGAN",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "train_rows": len(train),
    "synthetic_rows": len(synthetic_ctgan),
    "training_seconds": round(training_seconds, 2),
    "sdv_version": sdv.__version__,
    "torch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "training_csv": "data/processed/train_semantic.csv",
    "test_data_used_in_training": False,
}
with open(RUN_PATH, "w", encoding="utf-8") as file:
    json.dump(run_info, file, indent=2)

print("Saved:", MODEL_PATH)
print("Saved:", CSV_PATH)
print("Saved:", METADATA_PATH)
print("Saved:", RUN_PATH)

## 5. Initial synthetic-data checks

Compare readmission prevalence, mean prior healthcare visits, and category coverage against the real training data. These are **diagnostics**, not the final fidelity, utility, or privacy evaluation; the held-out test set will be used later for downstream utility analysis.

In [ ]:
print("Real readmission (%):", round((train[TARGET] == "1").mean() * 100, 3))
print("CTGAN readmission (%):", round((synthetic_ctgan[TARGET].astype(str) == "1").mean() * 100, 3))

visit_cols = ["number_outpatient", "number_emergency", "number_inpatient"]
print("\nMean prior visits:")
print(pd.DataFrame({
    "Real": train[visit_cols].mean(),
    "CTGAN": synthetic_ctgan[visit_cols].apply(pd.to_numeric, errors="coerce").mean(),
}).round(3))

print("\nCategory coverage:")
for col in ["race", "diag_3"]:
    print(f"{col}: real={train[col].nunique()}, CTGAN={synthetic_ctgan[col].nunique()}")

### Interpretation of CTGAN Results

CTGAN completed 300 training epochs and generated 79,567 synthetic clinical records.

The real training dataset contained 11.40% positive 30-day readmission cases, whereas the CTGAN-generated dataset contained 4.12%. This indicates that CTGAN reproduced the minority class but underestimated its prevalence.

CTGAN preserved all six race categories and all 20 `diag_3` diagnosis categories observed in the training data.

However, differences remained in prior healthcare utilization. CTGAN underestimated mean outpatient and emergency visits while overestimating mean inpatient visits (1.388 compared with 0.635 in the real data).

These initial findings suggest that CTGAN preserves categorical diversity but does not fully reproduce the numerical distributions and minority-outcome prevalence of the real clinical dataset.

Further evaluation of statistical fidelity, predictive utility, and privacy risk is required before drawing an overall conclusion about model performance.